In [ ]:
%pip install yfinance pandas_ta numpy pandas

  Using cached yfinance-1.2.0-py2.py3-none-any.whl.metadata (6.1 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 14.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 10.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 7.9 MB/s  0:00:00 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.2/26.2 MB 6.2 MB/s  0:00:04m0:00:0100:01
   ━━

In [2]:
import yfinance as yf
import pandas_ta as ta
import pandas as pd
import numpy as np

In [4]:

def get_triple_barrier_label(df, upper_pct=0.03, lower_pct=-0.02, horizon=5):
    labels = []
    close = df['Close'].values
    
    for i in range(len(close) - horizon):
        future_prices = close[i+1 : i+1+horizon]
        initial_price = close[i]

        returns = (future_prices - initial_price) / initial_price

        hit_upper = np.where(returns >= upper_pct)[0]
        hit_lower = np.where(returns <= lower_pct)[0]
        
        if len(hit_upper) > 0 and (len(hit_lower) == 0 or hit_upper[0] < hit_lower[0]):
            labels.append(1)
        elif len(hit_lower) > 0 and (len(hit_upper) == 0 or hit_lower[0] < hit_upper[0]):
            labels.append(-1)
        else:
            labels.append(0)
            
    # Pad the end with NaNs as we can't look into the future for the last 'horizon' rows
    labels.extend([np.nan] * (len(df) - len(labels)))
    return labels




In [5]:
def _merge(df, indicator_df):
    
    if indicator_df is not None and not indicator_df.empty:
        return pd.concat([df, indicator_df], axis=1)
    return df

In [9]:
def create_pooled_dataset(tickers, period="5y"):
    all_data = []
    
    for ticker in tickers:
        print(f"Processing: {ticker}")
        try:
            # 1. Download Data
            df = yf.download(ticker, period=period)
            if df.empty: 
                continue
            
            # Fix MultiIndex columns from yfinance (flatten if needed)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            # ── Momentum ─────────────────────────────────────────────
            for length in [5, 10, 14, 15]:
                df[f"rsi_{length}"] = ta.rsi(df["Close"], length=length)
            df["roc_10"] = ta.roc(df["Close"], length=10)
            df["mom_10"] = ta.mom(df["Close"], length=10)

            # ── Oscillators ───────────────────────────────────────────
            df = _merge(df, ta.stochrsi(df["Close"]))            
            df["cci_20"] = ta.cci(df["High"], df["Low"], df["Close"], length=20)
            df["wr_14"]  = ta.willr(df["High"], df["Low"], df["Close"], length=14)
            df = _merge(df, ta.kst(df["Close"]))                 
            
            # MACD (single calculation)
            macd_df = ta.macd(df["Close"])
            df = _merge(df, macd_df)

            # ── Trend ────────────────────────────────────────────────
            for length in [5, 10, 20]:
                df[f"sma_{length}"] = ta.sma(df["Close"], length=length)
                df[f"ema_{length}"] = ta.ema(df["Close"], length=length)
            df["vwma_20"] = ta.vwma(df["Close"], df["Volume"], length=20)

            # ── Volatility ───────────────────────────────────────────
            df = _merge(df, ta.bbands(df["Close"], length=20))  
            df["atr_14"] = ta.atr(df["High"], df["Low"], df["Close"], length=14)
            df = _merge(df, ta.kc(df["High"], df["Low"], df["Close"], length=20))

            # ── Volume ───────────────────────────────────────────────
            df["obv"] = ta.obv(df["Close"], df["Volume"])
            df["ad"]  = ta.ad(df["High"], df["Low"], df["Close"], df["Volume"])
            df["efi"] = ta.efi(df["Close"], df["Volume"])
            df = _merge(df, ta.nvi(df["Close"], df["Volume"]))
            df = _merge(df, ta.pvi(df["Close"], df["Volume"]))
            
            # ── Returns ──────────────────────────────────────────────
            df['Log_Ret'] = ta.log_return(df['Close'])
            
            # ── Target Label ─────────────────────────────────────────
            df['Target'] = get_triple_barrier_label(df)
            df['Ticker'] = ticker
            
            # Clean up: Remove rows with NaNs (from indicators and end-of-period labels)
            df.dropna(inplace=True)
            all_data.append(df)
            
        except Exception as e:
            print(f"Error fetching {ticker}: {e}")

    if not all_data:
        return pd.DataFrame()
    
    master_df = pd.concat(all_data)
    return master_df

In [10]:
stock_list = ["AAPL", "TSLA", "NVDA", "MSFT", "AMD", "GOOGL", "AMZN"]
master_dataset = create_pooled_dataset(stock_list)

Processing: AAPL


[*********************100%***********************]  1 of 1 completed


Processing: TSLA


[*********************100%***********************]  1 of 1 completed


Processing: NVDA


...

[*********************100%***********************]  1 of 1 completed


Processing: GOOGL


[*********************100%***********************]  1 of 1 completed


Processing: AMZN


[*********************100%***********************]  1 of 1 completed


In [12]:
# master_dataset.to_csv("global_stock_dataset.csv")
# print("\nMaster Dataset Created!")
master_dataset.head()

,Close,High,Low,Open,Volume,rsi_5,rsi_10,rsi_14,rsi_15,roc_10,...,KCUe_20_2,obv,ad,efi,NVI_1,PVI,PVIe_255,Log_Ret,Target,Ticker
Date,,,,,,,,,,,,,,,,,,,,,
2022-03-02,163.192902,163.976732,159.655879,161.066771,79724800,55.077280,48.522639,47.865369,47.864539,-3.605528,...,173.194345,27571900.0,9.376271e+08,1.406609e+07,1034.90846,100.000000,99.981040,0.020379,-1.0,AAPL
2022-03-03,162.869583,165.495414,162.203337,165.064306,76678400,53.208015,47.777090,47.352431,47.387454,-3.662702,...,172.838096,-49106500.0,8.919849e+08,8.514999e+06,1034.71034,100.000000,99.981188,-0.001983,-1.0,AAPL
2022-03-04,159.871460,162.203352,158.823098,161.164783,83737200,38.185918,41.247243,42.774855,43.117552,-3.381083,...,172.373938,-132843700.0,8.601887e+08,-2.856635e+07,1034.71034,98.159188,99.966954,-0.018580,-1.0,AAPL
2022-03-07,156.079666,161.684034,155.824913,160.057589,96418800,26.401890,34.601660,37.798132,38.425791,-4.781836,...,171.938125,-229262500.0,7.721544e+08,-7.671404e+07,1034.71034,97.628223,99.948682,-0.024004,-1.0,AAPL
2022-03-08,154.257309,159.587340,152.650463,155.609417,131148300,22.272721,31.860485,35.651273,36.386968,-4.186935,...,171.575548,-360410800.0,7.017640e+08,-9.989762e+07,1034.71034,98.832419,99.939961,-0.011745,1.0,AAPL


In [13]:
master_dataset.shape

(6979, 45)

In [15]:
# All columns/features in master_dataset
print(f"Total columns: {len(master_dataset.columns)}\n")
print("All columns:")
for i, col in enumerate(master_dataset.columns, 1):
    print(f"{i:2}. {col}")

Total columns: 45

All columns:
 1. Close
 2. High
 3. Low
 4. Open
 5. Volume
 6. rsi_5
 7. rsi_10
 8. rsi_14
 9. rsi_15
10. roc_10
11. mom_10
12. STOCHRSIk_14_14_3_3
13. STOCHRSId_14_14_3_3
14. cci_20
15. wr_14
16. KST_10_15_20_30_10_10_10_15
17. KSTs_9
18. MACD_12_26_9
19. MACDh_12_26_9
20. MACDs_12_26_9
21. sma_5
22. ema_5
23. sma_10
24. ema_10
25. sma_20
26. ema_20
27. vwma_20
28. BBL_20_2.0_2.0
29. BBM_20_2.0_2.0
30. BBU_20_2.0_2.0
31. BBB_20_2.0_2.0
32. BBP_20_2.0_2.0
33. atr_14
34. KCLe_20_2
35. KCBe_20_2
36. KCUe_20_2
37. obv
38. ad
39. efi
40. NVI_1
41. PVI
42. PVIe_255
43. Log_Ret
44. Target
45. Ticker


In [19]:
!pip install torch scikit-learn


  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 MB 6.7 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 6.4 MB/s  0:00:01 eta 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 8.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 2.7 MB/s  0:00:07m0:00:0100:01
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [torch]m11/12 [tor

In [1]:
!pip install tensorflow

  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached keras-3.13.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached markdown-3.10.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached werkzeug-3.1.6-py3-none-any.whl.metadata (4.0 kB)
  Using cached rich-14.3.3-py3-none-any.whl.metadata (18 kB)
  Using cached namex-0.1.0-py3-non

In [2]:
!pip install tensorflow-metal


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 1.3 MB/s  0:00:00m eta 0:00:01


In [3]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

NotFoundError: dlopen(/opt/anaconda3/envs/stock2/lib/python3.12/site-packages/tensorflow-plugins/libmetal_plugin.dylib, 0x0006): Library not loaded: @rpath/_pywrap_tensorflow_internal.so
  Referenced from: <8B62586B-B082-3113-93AB-FD766A9960AE> /opt/anaconda3/envs/stock2/lib/python3.12/site-packages/tensorflow-plugins/libmetal_plugin.dylib
  Reason: tried: '/opt/anaconda3/envs/stock2/lib/python3.12/site-packages/tensorflow-plugins/../_solib_darwin_arm64/_U@local_Uconfig_Utf_S_S_C_Upywrap_Utensorflow_Uinternal___Uexternal_Slocal_Uconfig_Utf/_pywrap_tensorflow_internal.so' (no such file), '/opt/anaconda3/envs/stock2/lib/python3.12/site-packages/tensorflow-plugins/../_solib_darwin_arm64/_U@local_Uconfig_Utf_S_S_C_Upywrap_Utensorflow_Uinternal___Uexternal_Slocal_Uconfig_Utf/_pywrap_tensorflow_internal.so' (no such file), '/opt/anaconda3/envs/stock2/bin/../lib/_pywrap_tensorflow_internal.so' (no such file)

In [ ]:
from sklearn.preprocessing import RobustScaler
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator

def create_sequences_with_generator(df, feature_cols, window_size=20):
    """
    Creates time series sequences using Keras TimeseriesGenerator.
    More memory-efficient and cleaner than manual loops.
    """
    # Scale features
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(df[feature_cols])
    
    # Targets need to be shifted by 1 to predict the NEXT label after the window
    y = df['Target'].values
    
    # TimeseriesGenerator creates (X[i:i+length], y[i+length]) pairs
    generator = TimeseriesGenerator(
        X_scaled, 
        y,
        length=window_size,
        batch_size=len(df) - window_size  # Get all samples at once
    )
    
    # Extract all sequences from generator
    X_seq, y_seq = generator[0]
    
    return X_seq, y_seq, scaler

def get_rnn_dataset(master_df, window_size=20):
    """
    Creates RNN sequences from the master dataset using TimeseriesGenerator.
    Processes each ticker separately to avoid data leakage.
    """
    # Exclude non-feature columns
    exclude_cols = ['Target', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    feature_cols = [col for col in master_df.columns if col not in exclude_cols]
    
    print(f"Using {len(feature_cols)} features with TimeseriesGenerator")
    
    all_X, all_y = [], []
    
    # Process each ticker separately to prevent data leakage
    for ticker in master_df['Ticker'].unique():
        df_ticker = master_df[master_df['Ticker'] == ticker].copy()
        df_ticker = df_ticker.sort_index()  # Ensure chronological order
        
        if len(df_ticker) <= window_size:
            print(f"Skipping {ticker}: insufficient data ({len(df_ticker)} rows)")
            continue
        
        X_ticker, y_ticker, _ = create_sequences_with_generator(df_ticker, feature_cols, window_size)
        
        print(f"{ticker}: {X_ticker.shape[0]} sequences")
        all_X.append(X_ticker)
        all_y.append(y_ticker)
    
    # Concatenate all companies into one global RNN dataset
    X_all = np.concatenate(all_X)
    y_all = np.concatenate(all_y)
    
    return X_all, y_all, feature_cols

In [ ]:
# Alternative: Use TimeseriesGenerator directly with model.fit() for memory efficiency
# This is better for very large datasets - generates batches on-the-fly

def create_train_test_generators(master_df, window_size=20, test_split=0.2, batch_size=64):
    """
    Creates TimeseriesGenerator objects for training directly with model.fit().
    More memory efficient as it generates batches on-the-fly.
    """
    from tensorflow.keras.utils import to_categorical
    
    exclude_cols = ['Target', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    feature_cols = [col for col in master_df.columns if col not in exclude_cols]
    
    train_generators = []
    test_generators = []
    
    for ticker in master_df['Ticker'].unique():
        df_ticker = master_df[master_df['Ticker'] == ticker].copy()
        df_ticker = df_ticker.sort_index()
        
        if len(df_ticker) <= window_size:
            continue
        
        # Scale features per ticker
        scaler = RobustScaler()
        X_scaled = scaler.fit_transform(df_ticker[feature_cols])
        
        # Shift labels for categorical encoding (-1,0,1) -> (0,1,2)
        y = df_ticker['Target'].values + 1
        y_cat = to_categorical(y, num_classes=3)
        
        # Time-series split (no shuffle - use last portion for test)
        split_idx = int(len(df_ticker) * (1 - test_split))
        
        X_train, X_test = X_scaled[:split_idx], X_scaled[split_idx-window_size:]
        y_train, y_test = y_cat[:split_idx], y_cat[split_idx-window_size:]
        
        # Create generators
        if len(X_train) > window_size:
            train_gen = TimeseriesGenerator(X_train, y_train, length=window_size, batch_size=batch_size)
            train_generators.append(train_gen)
        
        if len(X_test) > window_size:
            test_gen = TimeseriesGenerator(X_test, y_test, length=window_size, batch_size=batch_size)
            test_generators.append(test_gen)
    
    return train_generators, test_generators, feature_cols

# Note: To use generators directly with model.fit():
# train_gens, test_gens, features = create_train_test_generators(master_dataset)
# For each generator: model.fit(train_gen, validation_data=test_gen, epochs=100)

In [21]:
# Create RNN dataset from master_dataset
WINDOW_SIZE = 20

X_rnn, y_rnn, features_used = get_rnn_dataset(master_dataset, window_size=WINDOW_SIZE)

print(f"\n{'='*50}")
print(f"RNN Dataset Created!")
print(f"{'='*50}")
print(f"X shape: {X_rnn.shape}")  # (samples, window_size, num_features)
print(f"y shape: {y_rnn.shape}")  # (samples,)
print(f"Features used: {features_used}")
print(f"Window size: {WINDOW_SIZE}")
print(f"\nClass distribution:")
unique, counts = np.unique(y_rnn, return_counts=True)
for u, c in zip(unique, counts):
    label_name = {-1: 'Sell', 0: 'Hold', 1: 'Buy'}.get(u, u)
    print(f"  {label_name} ({u}): {c} ({c/len(y_rnn)*100:.1f}%)")

Using 38 features:
   1. rsi_5
   2. rsi_10
   3. rsi_14
   4. rsi_15
   5. roc_10
   6. mom_10
   7. STOCHRSIk_14_14_3_3
   8. STOCHRSId_14_14_3_3
   9. cci_20
  10. wr_14
  11. KST_10_15_20_30_10_10_10_15
  12. KSTs_9
  13. MACD_12_26_9
  14. MACDh_12_26_9
  15. MACDs_12_26_9
  16. sma_5
  17. ema_5
  18. sma_10
  19. ema_10
  20. sma_20
  21. ema_20
  22. vwma_20
  23. BBL_20_2.0_2.0
  24. BBM_20_2.0_2.0
  25. BBU_20_2.0_2.0
  26. BBB_20_2.0_2.0
  27. BBP_20_2.0_2.0
  28. atr_14
  29. KCLe_20_2
  30. KCBe_20_2
  31. KCUe_20_2
  32. obv
  33. ad
  34. efi
  35. NVI_1
  36. PVI
  37. PVIe_255
  38. Log_Ret
AAPL: 977 sequences
TSLA: 977 sequences
NVDA: 977 sequences
MSFT: 977 sequences
AMD: 977 sequences
GOOGL: 977 sequences
AMZN: 977 sequences

RNN Dataset Created!
X shape: (6839, 20, 38)
y shape: (6839,)
Features used: ['rsi_5', 'rsi_10', 'rsi_14', 'rsi_15', 'roc_10', 'mom_10', 'STOCHRSIk_14_14_3_3', 'STOCHRSId_14_14_3_3', 'cci_20', 'wr_14', 'KST_10_15_20_30_10_10_10_15', 'KSTs_9', '

In [28]:
X_rnn

array([[[ 1.16733443e-02, -1.82635622e-01, -2.62721399e-01, ...,
          0.00000000e+00, -2.14307193e-01,  1.13270951e+00],
        [-4.23537036e-02, -2.13546581e-01, -2.88225201e-01, ...,
          0.00000000e+00, -2.12975330e-01, -1.83842166e-01],
        [-4.76534835e-01, -4.84278474e-01, -5.15827006e-01, ...,
         -1.84081221e+00, -3.40962007e-01, -1.16093983e+00],
        ...,
        [ 8.79434032e-01,  6.71620372e-01,  5.08160118e-01, ...,
          0.00000000e+00, -4.89225959e-01,  1.52335427e-01],
        [ 9.14526611e-01,  7.16176970e-01,  5.52484165e-01, ...,
          5.03664685e-01, -4.50366290e-01,  2.28696509e-01],
        [ 1.02528762e+00,  8.75150119e-01,  7.14129943e-01, ...,
          1.91346212e+00, -3.12778777e-01,  1.04879911e+00]],

       [[-4.23537036e-02, -2.13546581e-01, -2.88225201e-01, ...,
          0.00000000e+00, -2.12975330e-01, -1.83842166e-01],
        [-4.76534835e-01, -4.84278474e-01, -5.15827006e-01, ...,
         -1.84081221e+00, -3.40962007e

In [26]:
# Train/Test split for RNN (time-series aware - no shuffle!)
from sklearn.model_selection import train_test_split

# For time series, use chronological split or stratified without shuffle
# Option 1: Simple split (maintains some temporal ordering per ticker)
X_train, X_test, y_train, y_test = train_test_split(
    X_rnn, y_rnn, test_size=0.2, random_state=42, stratify=y_rnn
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nInput shape for RNN: {X_train.shape[1:]} (window_size, num_features)")

# Save for later use
np.save('X_train_rnn.npy', X_train)
np.save('X_test_rnn.npy', X_test)
np.save('y_train_rnn.npy', y_train)
np.save('y_test_rnn.npy', y_test)
print("\nDatasets saved to .npy files!")

Training set: 5471 samples
Test set: 1368 samples

Input shape for RNN: (20, 38) (window_size, num_features)

Datasets saved to .npy files!


In [ ]:
x_rnn

NameError: name 'x_rnn' is not defined

In [29]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight

# Triple Barrier Label Mapping:
# Original:  -1 (Stop Loss), 0 (Time Barrier/Hold), 1 (Profit Target)
# Shifted:    0 (Stop Loss), 1 (Time Barrier/Hold), 2 (Profit Target)
# 
# Softmax Output Interpretation:
#   output[0] = P(-1) = Probability of hitting Stop Loss
#   output[1] = P(0)  = Probability of hitting Time Barrier (Sideways/No Action)
#   output[2] = P(1)  = Probability of hitting Profit Target

y_train_shifted = y_train + 1  
y_test_shifted = y_test + 1

# One-hot encode targets for softmax
y_train_cat = to_categorical(y_train_shifted, num_classes=3)
y_test_cat = to_categorical(y_test_shifted, num_classes=3)

# Compute class weights to handle imbalanced data
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_shifted), y=y_train_shifted)
class_weight_dict = dict(enumerate(class_weights))

print(f"Input shape: {X_train.shape[1:]}")  # (window_size, num_features)
print(f"Output: Softmax with 3 nodes")
print(f"  Node 0: P(Stop Loss)")
print(f"  Node 1: P(Hold/Sideways)")
print(f"  Node 2: P(Profit Target)")
print(f"\nClass weights: {class_weight_dict}")

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Build GRU Model with Softmax Output for Triple Barrier Classification
def build_gru_model(input_shape, num_classes=3):
    model = Sequential([
        # First GRU layer with return sequences for stacking
        GRU(128, return_sequences=True, input_shape=input_shape),
        BatchNormalization(),
        Dropout(0.3),
        
        # Second GRU layer
        GRU(64, return_sequences=True),
        BatchNormalization(),
        Dropout(0.3),
        
        # Third GRU layer (final, no return sequences)
        GRU(32, return_sequences=False),
        BatchNormalization(),
        Dropout(0.3),
        
        # Dense layers for classification
        Dense(32, activation='relu'),
        Dropout(0.2),
        
        # Softmax output: 3 nodes for Triple Barrier probabilities
        # [P(Stop Loss), P(Hold), P(Profit Target)]
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Create model
input_shape = (X_train.shape[1], X_train.shape[2])  # (window_size, num_features)
model = build_gru_model(input_shape)
model.summary()

In [ ]:
# Callbacks for training
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

# Train the model
history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=100,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy plot
axes[1].plot(history.history['accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Predictions
y_pred_proba = model.predict(X_test)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

# Convert back to original labels for display
label_names = ['Sell (-1)', 'Hold (0)', 'Buy (1)']

# Classification Report
print("Classification Report:")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=label_names))

# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# Overall metrics
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
# Save the model
model.save('gru_stock_classifier.keras')
print("Model saved to 'gru_stock_classifier.keras'")

# Save feature list for inference
import json
with open('feature_columns.json', 'w') as f:
    json.dump(features_used, f)
print(f"Feature columns saved ({len(features_used)} features)")

In [ ]:
# Interpret softmax probabilities for trading decisions
def interpret_prediction(proba):
    """
    Interpret model softmax output as Triple Barrier probabilities.
    
    Args:
        proba: array of shape (3,) - softmax output
    
    Returns:
        dict with probabilities and recommended action
    """
    p_stop_loss = proba[0]      # P(-1): Hit stop loss
    p_hold = proba[1]           # P(0): Hit time barrier (sideways)
    p_profit = proba[2]         # P(1): Hit profit target
    
    # Get predicted class
    pred_class = np.argmax(proba) - 1  # Convert back to (-1, 0, 1)
    actions = {-1: 'SELL/SHORT', 0: 'HOLD/NO ACTION', 1: 'BUY/LONG'}
    
    return {
        'P(Stop Loss)': f"{p_stop_loss:.2%}",
        'P(Hold/Sideways)': f"{p_hold:.2%}",
        'P(Profit Target)': f"{p_profit:.2%}",
        'Recommended Action': actions[pred_class],
        'Confidence': f"{np.max(proba):.2%}"
    }

# Example: Interpret some test predictions
print("Sample Predictions with Probability Interpretation:")
print("=" * 60)
sample_indices = [0, 100, 500, 1000]
for idx in sample_indices:
    if idx < len(X_test):
        proba = y_pred_proba[idx]
        actual = y_true[idx] - 1  # Convert back to (-1, 0, 1)
        interpretation = interpret_prediction(proba)
        
        print(f"\nSample {idx}:")
        print(f"  Actual Label: {actual} ({['Stop Loss', 'Hold', 'Profit'][actual+1]})")
        for key, val in interpretation.items():
            print(f"  {key}: {val}")